# OpicTutor — OmniVoice로 진행자 음성 만들기

> AI-assisted: this notebook was written with help from Claude (Anthropic).

[OmniVoice](https://github.com/k2-fsa/OmniVoice)(Apache-2.0)로 시험 문항과 진행자 대사를 mp3로 미리 만듭니다.

**순서**
1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. 로컬에서 `python -m scripts.export_tts_jobs`로 만든 `backend/static/tts/jobs.jsonl` 준비
3. 위에서부터 셀을 차례로 실행
4. 마지막 셀에서 받은 `tts_mp3.zip`을 로컬에서 `python -m scripts.import_tts tts_mp3.zip --activate`로 반입(압축을 직접 풀지 않음)

⚠️ 실존 인물의 목소리를 복제하지 않습니다. 기준 음성은 Voice Design으로 새로 만듭니다.

> 2026-11 갱신(tts-host): 출력 경로 `{kind}/{ref_id}.mp3`, manifest에 `model_revision`·`settings_version` 추가, zip 최상위 폴더 = `settings_version`(= voice_id), e2e 답변용 별도 목소리 셀(선택).

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. 설치

Colab에 이미 들어 있는 PyTorch를 그대로 씁니다. 설치 후 버전 충돌 경고가 나오면 **런타임 → 세션 다시 시작** 후 3번 셀부터 실행하세요.

In [ ]:
%pip install -q omnivoice soundfile

## 3. 설정

- `INSTRUCT`: 목소리 설계 속성. 카테고리마다 하나씩, 쉼표로 구분합니다.
  - 성별 `male` / `female`
  - 나이 `young adult` / `middle-aged` …
  - 음높이 `low pitch` / `moderate pitch` / `high pitch` …
  - 억양 `american accent` / `british accent` …
- `SEED`를 바꾸면 다른 목소리가 나옵니다. 마음에 드는 목소리가 나오면 그 값을 기록해 두세요.

In [ ]:
INSTRUCT = "female, young adult, moderate pitch, american accent"
SEED = 42
SPEED = 0.95        # 1.0보다 작으면 느리게 (시험 진행자 톤)
NUM_STEP = 32       # 확산 단계 수. 16으로 줄이면 빨라지지만 품질이 떨어질 수 있음
MP3_BITRATE = "64k"
SAMPLE_RATE = 24000 # OmniVoice 출력 고정값

# 기준 음성용 문장 (3~10초 분량이 가장 좋음)
REF_TEXT = "Hi there. I'll be asking you a few questions today, so just relax and answer naturally."

## 4. jobs.jsonl 업로드

In [ ]:
import json, pathlib
from google.colab import files

uploaded = files.upload()   # backend/static/tts/jobs.jsonl 선택
name = next(iter(uploaded))
jobs = [json.loads(l) for l in open(name, encoding="utf-8") if l.strip()]

ids = [j["id"] for j in jobs]
assert len(ids) == len(set(ids)), "jobs.jsonl에 중복 id가 있습니다"
print(f"{len(jobs)}개 작업, 총 {sum(len(j['text']) for j in jobs)}자")
for j in jobs[:3]:
    print(" -", j["id"], "|", j["text"][:70])

## 5. 모델 불러오기 (처음 한 번은 Hugging Face에서 내려받음)

In [ ]:
import torch
from omnivoice import OmniVoice

MODEL_ID = "k2-fsa/OmniVoice"
try:
    from huggingface_hub import HfApi
    MODEL_REVISION = HfApi().model_info(MODEL_ID).sha or "unknown"
except Exception as e:  # 조회 실패해도 생성은 진행
    print("model_revision 조회 실패:", e)
    MODEL_REVISION = "unknown"

# 기존과 같은 방식으로 불러오고, 그 시점의 최신 커밋 sha를 manifest에 기록한다
model = OmniVoice.from_pretrained(MODEL_ID, device_map="cuda:0", dtype=torch.float16)
print("loaded", MODEL_REVISION)

## 6. 기준 음성 설계 (Voice Design)

들어 보고 마음에 들지 않으면 3번 셀의 `SEED`나 `INSTRUCT`를 바꾼 뒤 3번 셀과 이 셀만 다시 실행하세요.

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

torch.manual_seed(SEED)
ref = model.generate(text=REF_TEXT, instruct=INSTRUCT, speed=SPEED, num_step=NUM_STEP)[0]
sf.write("ref_voice.wav", ref, SAMPLE_RATE)
print(f"{len(ref) / SAMPLE_RATE:.1f}초")
display(Audio("ref_voice.wav"))

## 7. 목소리 고정 (Voice Clone Prompt)

Voice Design만으로 문장을 하나씩 만들면 문장마다 목소리가 조금씩 달라집니다. 6번에서 만든 기준 음성을 프롬프트로 저장해 모든 문장에 같은 목소리를 씁니다.

In [ ]:
prompt = model.create_voice_clone_prompt(ref_audio="ref_voice.wav", ref_text=REF_TEXT)
prompt.save("host_voice.pt")
print("saved host_voice.pt")

## 8. 전체 생성 → mp3 변환

In [ ]:
import hashlib, subprocess, time

# contracts.yaml data_files.tts_manifest: settings_version = sha1(정렬 키 JSON of 6개 값)[:10] = voice_id
SETTINGS = {"engine": "k2-fsa/OmniVoice", "model_revision": MODEL_REVISION, "instruct": INSTRUCT,
            "seed": SEED, "speed": SPEED, "num_step": NUM_STEP}
SETTINGS_VERSION = hashlib.sha1(json.dumps(SETTINGS, sort_keys=True, ensure_ascii=False,
                                           separators=(",", ":")).encode("utf-8")).hexdigest()[:10]
out = pathlib.Path("tts_out") / SETTINGS_VERSION      # zip 최상위 폴더 = settings_version
manifest = []
t0 = time.time()

host_jobs = [j for j in jobs if j["kind"] != "e2e_answer"]   # e2e 답변은 11번 셀(다른 목소리)에서
for i, job in enumerate(host_jobs, 1):
    mp3 = out / job["kind"] / f"{job['ref_id']}.mp3"          # {kind}/{ref_id}.mp3
    wav = mp3.with_suffix(".wav")
    mp3.parent.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(SEED)
    audio = model.generate(text=job["text"], voice_clone_prompt=prompt,
                           speed=SPEED, num_step=NUM_STEP)[0]
    sf.write(wav, audio, SAMPLE_RATE)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(wav),
                    "-ac", "1", "-b:a", MP3_BITRATE, str(mp3)], check=True)
    wav.unlink()

    manifest.append({
        "id": job["id"],
        "text_sha1": job.get("text_sha1") or hashlib.sha1(job["text"].strip().encode("utf-8")).hexdigest(),
        "seconds": round(len(audio) / SAMPLE_RATE, 2),
    })
    print(f"[{i}/{len(host_jobs)}] {job['id']}  {manifest[-1]['seconds']}s")

def write_manifest():
    (out / "manifest.json").write_text(json.dumps({**SETTINGS, "settings_version": SETTINGS_VERSION,
                                                   "items": manifest}, ensure_ascii=False, indent=2), encoding="utf-8")
write_manifest()
print(f"voice_id(settings_version) = {SETTINGS_VERSION}, done in {time.time() - t0:.0f}s")

## 9. 미리 듣기

In [ ]:
for job in host_jobs[:3]:
    print(job["id"], "|", job["text"][:80])
    display(Audio(str(out / job["kind"] / f"{job['ref_id']}.mp3")))

## 11. (선택) e2e 답변 음성 — 진행자와 **다른** 목소리

`export_tts_jobs --e2e`로 만든 `e2e_answer` 항목(자동 완주 테스트용 "가짜 수험생 음성", 실제 사람 목소리 아님)이 있을 때만 실행합니다.
진행자와 섞이지 않도록 다른 `E2E_INSTRUCT`로 기준 음성을 새로 설계해 별도 Voice Clone Prompt를 씁니다. 결과는 같은 manifest에 추가됩니다.

In [ ]:
E2E_INSTRUCT = "male, young adult, low pitch, korean accent"
E2E_SEED = 7
e2e_jobs = [j for j in jobs if j["kind"] == "e2e_answer"]
if e2e_jobs:
    torch.manual_seed(E2E_SEED)
    e2e_ref = model.generate(text=REF_TEXT, instruct=E2E_INSTRUCT, speed=1.0, num_step=NUM_STEP)[0]
    sf.write("e2e_ref.wav", e2e_ref, SAMPLE_RATE)
    e2e_prompt = model.create_voice_clone_prompt(ref_audio="e2e_ref.wav", ref_text=REF_TEXT)
    for i, job in enumerate(e2e_jobs, 1):
        mp3 = out / job["kind"] / f"{job['ref_id']}.mp3"
        wav = mp3.with_suffix(".wav"); mp3.parent.mkdir(parents=True, exist_ok=True)
        torch.manual_seed(E2E_SEED)
        audio = model.generate(text=job["text"], voice_clone_prompt=e2e_prompt, speed=1.0, num_step=NUM_STEP)[0]
        sf.write(wav, audio, SAMPLE_RATE)
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(wav), "-ac", "1", "-b:a", MP3_BITRATE, str(mp3)], check=True)
        wav.unlink()
        manifest.append({"id": job["id"], "text_sha1": job["text_sha1"], "seconds": round(len(audio) / SAMPLE_RATE, 2)})
        print(f"[e2e {i}/{len(e2e_jobs)}] {job['id']}")
    write_manifest()
else:
    print("e2e_answer 항목 없음 — 건너뜀")

## 10. 다운로드

받은 `tts_mp3.zip`은 **압축을 풀지 말고** 로컬에서 반입합니다.

```bash
cd backend
python -m scripts.import_tts tts_mp3.zip --activate
```

zip 최상위 폴더가 `settings_version`(= voice_id)이고 그 안에 `question/`, `host/`, (선택) `e2e_answer/`, `manifest.json`이 있습니다.
`host_voice.pt`도 함께 받아 두면 나중에 문장을 추가할 때(`export_tts_jobs --stale`) 같은 목소리를 쓸 수 있습니다
(6번 셀 대신 `prompt = VoiceClonePrompt.load("host_voice.pt")`, 같은 `SEED`·`SPEED`·`NUM_STEP` 유지 → 같은 settings_version).

In [ ]:
import shutil
shutil.make_archive("tts_mp3", "zip", root_dir="tts_out", base_dir=SETTINGS_VERSION)
files.download("tts_mp3.zip")
files.download("host_voice.pt")